# PUDO Risk — Waymo CPUC Deployment Data (2024 Jun – 2026 Q2)

This notebook answers the three steps in the Fall 2026 class exercise:

1. identify sources of danger, harm, or loss;
2. determine the consequences of those hazards;
3. determine likelihood using CPUC complaints while controlling for miles driven.

It also checks whether the exposure-adjusted complaint frequency is rising or
falling and gives a transparent, provisional risk-chart placement.


In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
import polars as pl
from scipy import stats
import statsmodels.api as sm

DATA = Path(os.environ.get("PUDO_DATA_DIR", "../data")) / "parquet"
paths = {
    "summary": DATA / "pudo_summary.parquet",
    "monthly": DATA / "monthly_activity.parquet",
    "complaints": DATA / "complaints.parquet",
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Run pudo-build first; missing: {missing}")

summary = pl.read_parquet(paths["summary"])
monthly = pl.read_parquet(paths["monthly"])
complaints = pl.read_parquet(paths["complaints"])
summary


In [ ]:
# Notebook-side guard against stale or manually edited Parquet files.
required = [
    "pudo_complaints",
    "pudo_collisions",
    "vmt",
    "trips",
    "pudo_per_100k_vmt",
    "pudo_per_100k_trips",
    "time_months",
]
null_counts = summary.select(
    [pl.col(column).null_count().alias(column) for column in required]
).row(0)
bad = {column: count for column, count in zip(required, null_counts) if count}
if bad:
    raise ValueError(f"Summary contains null critical values: {bad}")
if summary.filter((pl.col("vmt") <= 0) | (pl.col("trips") <= 0)).height:
    raise ValueError("Summary contains non-positive exposure")
if summary.get_column("period_id").n_unique() != summary.height:
    raise ValueError("Summary contains duplicate reporting periods")

s = summary.sort("period_midpoint").to_pandas()
print(f"Validated {summary.height} reporting periods; no critical null values.")


## Reporting-period caveat: “8 quarters” versus 9 CPUC windows

The class slide calls Q3 2024–Q2 2026 “8 quarters.” CPUC did not publish the
2024 complaint numerator on calendar-quarter boundaries: it published
Jun–Aug, Sep–Nov, then a one-month Dec transition report before changing to
calendar quarters in 2025. Complaint timestamps are redacted, so the 2024
counts cannot be reassigned to calendar months without inventing data.

- The primary time-trend model therefore uses all **9 actual reporting
  windows**, their true midpoints, and an exposure offset.
- For the requested Y1/Y2 classroom comparison, Sep–Nov and Dec 2024 are
  combined into one explicitly labeled transition block, producing **4 blocks
  in Y1 and 4 in Y2** while retaining all observations.


In [ ]:
def combine_rows(label, rows, year_group):
    complaints_total = rows["pudo_complaints"].sum()
    collisions_total = rows["pudo_collisions"].sum()
    vmt_total = rows["vmt"].sum()
    trips_total = rows["trips"].sum()
    return {
        "block": label,
        "year_group": year_group,
        "pudo_complaints": complaints_total,
        "pudo_collisions": collisions_total,
        "vmt": vmt_total,
        "trips": trips_total,
        "pudo_per_100k_vmt": complaints_total / vmt_total * 100_000,
        "pudo_per_100k_trips": complaints_total / trips_total * 100_000,
    }


course_blocks = pd.DataFrame(
    [
        combine_rows("2024 Q3 report (Jun–Aug)", s.iloc[[0]], "Y1"),
        combine_rows("2024 Q4 transition (Sep–Dec)", s.iloc[1:3], "Y1"),
        combine_rows("2025 Q1", s.iloc[[3]], "Y1"),
        combine_rows("2025 Q2", s.iloc[[4]], "Y1"),
        combine_rows("2025 Q3", s.iloc[[5]], "Y2"),
        combine_rows("2025 Q4", s.iloc[[6]], "Y2"),
        combine_rows("2026 Q1", s.iloc[[7]], "Y2"),
        combine_rows("2026 Q2", s.iloc[[8]], "Y2"),
    ]
)
course_blocks


## Step 3A. Baseline statistics and distributions

Raw counts can increase simply because service grows. The primary exposure
metric requested in class is VMT; complaints per trip are retained as a
sensitivity measure because trips represent pickup/drop-off opportunities.


In [ ]:
metrics = [
    "pudo_complaints",
    "vmt",
    "trips",
    "pudo_per_100k_vmt",
    "pudo_per_100k_trips",
]
baseline = course_blocks[metrics].agg(["count", "mean", "std", "median", "min", "max"]).T
baseline


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
distribution_specs = [
    ("pudo_complaints", "PUDO complaints"),
    ("vmt", "VMT"),
    ("pudo_per_100k_vmt", "Complaints per 100k VMT"),
]
for ax, (column, title) in zip(axes, distribution_specs):
    ax.hist(course_blocks[column], bins="auto", color="#2471a3", alpha=0.8, edgecolor="white")
    ax.set_title(f"{title} (n=8 blocks)")
    ax.set_ylabel("Number of blocks")
plt.tight_layout()
plt.show()


## Step 3B. Y1 versus Y2 statistical comparisons

The class slides request both parametric and non-parametric comparisons.
Welch’s t-test does not assume equal variance; Mann–Whitney U is included as a
rank-based sensitivity test. With only four blocks per group, both tests have
low power and should be reported with the descriptive statistics, not as a
binary proof of “same” or “different.”


In [ ]:
test_rows = []
for metric in ["vmt", "pudo_complaints", "pudo_per_100k_vmt", "pudo_per_100k_trips"]:
    y1 = course_blocks.loc[course_blocks["year_group"] == "Y1", metric]
    y2 = course_blocks.loc[course_blocks["year_group"] == "Y2", metric]
    welch = stats.ttest_ind(y1, y2, equal_var=False)
    mann = stats.mannwhitneyu(y1, y2, alternative="two-sided")
    test_rows.append(
        {
            "metric": metric,
            "Y1 mean": y1.mean(),
            "Y2 mean": y2.mean(),
            "Welch t p": welch.pvalue,
            "Mann–Whitney p": mann.pvalue,
        }
    )

year_tests = pd.DataFrame(test_rows)
year_tests


In [ ]:
def poisson_rate_ratio(first, last, exposure):
    rate_ratio = (first["pudo_complaints"] / first[exposure]) / (
        last["pudo_complaints"] / last[exposure]
    )
    standard_error = np.sqrt(
        1 / first["pudo_complaints"] + 1 / last["pudo_complaints"]
    )
    low, high = np.exp(np.log(rate_ratio) + np.array([-1, 1]) * 1.96 * standard_error)
    z_value = np.log(rate_ratio) / standard_error
    p_value = 2 * stats.norm.sf(abs(z_value))
    return rate_ratio, low, high, p_value


endpoint_rows = []
for exposure in ["vmt", "trips"]:
    ratio, low, high, p_value = poisson_rate_ratio(s.iloc[5], s.iloc[8], exposure)
    endpoint_rows.append(
        {
            "comparison": f"2025 Q3 / 2026 Q2 by {exposure}",
            "rate ratio": ratio,
            "95% CI low": low,
            "95% CI high": high,
            "p": p_value,
        }
    )
pd.DataFrame(endpoint_rows)


## Step 3C. Is the exposure-adjusted frequency rising or falling?

Poisson regression uses complaint counts with log(exposure) as an offset and
the actual midpoint of each reporting window as time. Robust standard errors
are used. Because the complaint schema changes at 2025, a 2025+ sensitivity
model is essential: a full-window slope may otherwise measure a reporting
change rather than a safety change.


In [ ]:
def fit_poisson_trend(data, exposure, label):
    X = sm.add_constant(data[["time_months"]])
    model = sm.GLM(
        data["pudo_complaints"],
        X,
        family=sm.families.Poisson(),
        offset=np.log(data[exposure]),
    ).fit(cov_type="HC0")
    beta = model.params["time_months"]
    low, high = model.conf_int().loc["time_months"]
    annual_change = np.exp(beta * 12) - 1
    annual_ci = np.exp(np.array([low, high]) * 12) - 1
    dispersion = model.pearson_chi2 / model.df_resid
    print(
        f"{label}, {exposure}: {annual_change:+.1%}/year "
        f"(95% CI {annual_ci[0]:+.1%} to {annual_ci[1]:+.1%}), "
        f"p={model.pvalues['time_months']:.4f}, dispersion={dispersion:.2f}"
    )
    return model


print("All 9 reporting windows:")
full_vmt_model = fit_poisson_trend(s, "vmt", "full window")
full_trip_model = fit_poisson_trend(s, "trips", "full window")

post_change = s[s["period_id"].str.match(r"2025|2026")].copy()
print("\nComparable 2025+ row-level schema only:")
post_vmt_model = fit_poisson_trend(post_change, "vmt", "2025+")
post_trip_model = fit_poisson_trend(post_change, "trips", "2025+")


In [ ]:
def fit_negative_binomial(data, exposure):
    X = sm.add_constant(data[["time_months"]])
    model = sm.NegativeBinomial(
        data["pudo_complaints"], X, exposure=data[exposure]
    ).fit(disp=0, cov_type="HC0")
    beta = model.params["time_months"]
    low, high = model.conf_int().loc["time_months"]
    annual_change = np.exp(beta * 12) - 1
    annual_ci = np.exp(np.array([low, high]) * 12) - 1
    print(
        f"NB sensitivity, {exposure}: {annual_change:+.1%}/year "
        f"(95% CI {annual_ci[0]:+.1%} to {annual_ci[1]:+.1%}), "
        f"p={model.pvalues['time_months']:.4f}"
    )
    return model


nb_vmt_model = fit_negative_binomial(s, "vmt")
nb_trip_model = fit_negative_binomial(s, "trips")


In [ ]:
labels = s["period_id"]
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].bar(labels, s["pudo_complaints"], color="#b03a2e", label="PUDO complaints")
axes[0].set_ylabel("Raw count")
axes[0].legend()
axes[1].plot(labels, s["pudo_per_100k_vmt"], marker="o", label="per 100k VMT")
axes[1].plot(labels, s["pudo_per_100k_trips"], marker="s", label="per 100k trips")
axes[1].set_ylabel("Exposure-adjusted rate")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=55)
plt.tight_layout()
plt.show()


## Steps 1–2. Hazards and consequences

**Hazards (sources of danger, harm, or loss)**

- stopping in a travel lane, bike lane, intersection, or otherwise unsafe
  pickup/drop-off location;
- passenger entry/exit conflicts with moving vehicles, cyclists, or
  pedestrians;
- localization/routing failure that leaves passengers away from a safe curb;
- obstruction, unexpected stopping, or traffic-law violations around PUDO;
- accessibility failures that expose riders to extra roadway crossings or
  prevent a safe pickup.

**Consequences** range from inconvenience, delay, and accessibility loss to
traffic obstruction, property damage, injury, or death. The public CPUC files
redact incident time/location and do not provide a consistent injury-severity
scale. Therefore a complaint is not interchangeable with physical harm, and
PUDO collisions must be shown separately.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(labels, s["pudo_complaints"] / s["vmt"] * 1e5, marker="o", label="Complaints")
ax.plot(labels, s["pudo_collisions"] / s["vmt"] * 1e5, marker="^", label="PUDO collisions")
if s["pudo_travel_lane"].notna().any():
    ax.plot(
        labels,
        s["pudo_travel_lane"] / s["vmt"] * 1e5,
        marker="s",
        label="Travel-lane PUDO",
    )
ax.set_ylabel("Events per 100k VMT")
ax.tick_params(axis="x", rotation=55)
ax.legend()
plt.tight_layout()
plt.show()


## Provisional risk-chart placement

The lecture matrix uses five likelihood levels (Rare → Almost Certain) and
five impact levels (Insignificant → Severe), but supplies no numeric frequency
thresholds. Placement therefore depends on the event definition:

- **Exposure-normalized primary view:** a reported PUDO complaint is below 6
  per 100,000 trips in every period. Calling this **Rare (1)** and a typical
  complaint **Minor (2)** gives **Very Low 2**. Both labels are provisional;
  the class must agree on frequency and impact cutoffs.
- **Fleet/reporting-window sensitivity:** at least one PUDO complaint occurs
  in every reporting window, which is **Almost Certain (5)** at the fleet
  level. With Minor impact, that is **High 10**. This different answer shows
  why the unit of analysis must be stated.
- A collision with injury is a separate hazard and may have Significant,
  Major, or Severe consequence; the public fields used here do not support a
  defensible severity assignment.


In [ ]:
likelihood = ["Rare 1", "Unlikely 2", "Moderate 3", "Likely 4", "Almost Certain 5"]
impact = ["Insignificant 1", "Minor 2", "Significant 3", "Major 4", "Severe 5"]
labels_matrix = np.array(
    [
        ["Very low 1", "Very low 2", "Low 3", "Medium 4", "Medium 5"],
        ["Very low 2", "Low 4", "Medium 6", "Medium 8", "High 10"],
        ["Low 3", "Medium 6", "Medium 9", "High 12", "Very high 15"],
        ["Medium 4", "Medium 8", "High 12", "Very high 16", "Extreme 20"],
        ["Medium 5", "High 10", "Very high 15", "Extreme 20", "Extreme 25"],
    ]
)
category = np.array(
    [
        [0, 0, 1, 2, 2],
        [0, 1, 2, 2, 3],
        [1, 2, 2, 3, 4],
        [2, 2, 3, 4, 5],
        [2, 3, 4, 5, 5],
    ]
)
cmap = ListedColormap(["#59a14f", "#2f8f46", "#f1ce3e", "#f28e2b", "#e15759", "#b22222"])
fig, ax = plt.subplots(figsize=(11, 6))
ax.imshow(category, cmap=cmap, vmin=0, vmax=5, origin="lower")
for row in range(5):
    for column in range(5):
        ax.text(column, row, labels_matrix[row, column], ha="center", va="center", fontsize=9)
ax.scatter([1], [0], marker="o", s=220, facecolors="none", edgecolors="black", linewidths=2, label="Exposure-normalized provisional")
ax.scatter([1], [4], marker="s", s=190, facecolors="none", edgecolors="white", linewidths=2, label="Fleet-window sensitivity")
ax.set_xticks(range(5), impact, rotation=20, ha="right")
ax.set_yticks(range(5), likelihood)
ax.set_xlabel("Impact / consequence")
ax.set_ylabel("Likelihood")
ax.set_title("PUDO risk matrix — event definition changes placement")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()


## Conclusion to report

The raw complaint count rises as Waymo’s activity grows, but raw counts are not
the risk rate. Across all nine CPUC windows the exposure-adjusted model may
show a decline; however, the 2024 aggregate schema and 2025+ row-level schema
are not directly comparable. The 2025+ sensitivity model should drive the
trend wording. Report its direction, confidence interval, and p-value rather
than claiming a simple monotonic rise or fall.

The risk-chart cell is provisional because the lecture provides qualitative
labels without numeric cutoffs and because consequence severity is redacted.
State the event unit and keep complaint likelihood separate from collision
severity.
